In [1]:
import pyspark, os, sys, pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as F 
import pyspark.sql.types as T
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])
spark = SparkSession.builder.appName("spark-sql-preparation").getOrCreate()
sc = spark.sparkContext
print("Spark version:", spark.version)
spark

Python /usr/local/bin/python
Java /usr/lib/jvm/java-17-openjdk-arm64


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/12 14:25:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2


## Spark SQL

Spark SQL allows you to run standard SQL queries on DataFrames by registering them as  
**temporary views** with `df.createOrReplaceTempView("name")`.

Two equivalent APIs produce the same optimized execution plan:

```python
# DataFrame API
df.where(F.col("Origin") == "USA").count()

# Spark SQL
spark.sql('SELECT COUNT(*) FROM cars WHERE Origin = "USA"').first()[0]
```

**User-Defined Functions (UDFs)** extend SQL with custom Python logic.  
Register globally with `spark.udf.register("name", fn, return_type)` to use in SQL queries.

> Prefer built-in Spark functions (`F.upper`, `F.initcap`, …) over UDFs when possible --  
> built-ins run natively in the JVM and are much faster than Python UDFs.

# Example 1

## Cars Dataset

The `data/cars.json` file contains 406 classic cars with attributes:  
`Name`, `Miles_per_Gallon`, `Cylinders`, `Displacement`, `Horsepower`, `Weight_in_lbs`, `Acceleration`, `Year`, `Origin`

We register it as a temp view and then query it both ways -- showing that the DataFrame API  
and Spark SQL produce identical results (they share the same Catalyst optimizer).

In [2]:
file_name = os.path.join(os.getcwd(), "data/cars.json")
cars = spark.read.format("json").option("inferSchema", "true").load(file_name)
cars

DataFrame[Acceleration: double, Cylinders: bigint, Displacement: double, Horsepower: bigint, Miles_per_Gallon: double, Name: string, Origin: string, Weight_in_lbs: bigint, Year: string]

In [3]:
cars.printSchema()

root
 |-- Acceleration: double (nullable = true)
 |-- Cylinders: long (nullable = true)
 |-- Displacement: double (nullable = true)
 |-- Horsepower: long (nullable = true)
 |-- Miles_per_Gallon: double (nullable = true)
 |-- Name: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Weight_in_lbs: long (nullable = true)
 |-- Year: string (nullable = true)



In [4]:
cars.show(3)

+------------+---------+------------+----------+----------------+--------------------+------+-------------+----------+
|Acceleration|Cylinders|Displacement|Horsepower|Miles_per_Gallon|                Name|Origin|Weight_in_lbs|      Year|
+------------+---------+------------+----------+----------------+--------------------+------+-------------+----------+
|        12.0|        8|       307.0|       130|            18.0|chevrolet chevell...|   USA|         3504|1970-01-01|
|        11.5|        8|       350.0|       165|            15.0|   buick skylark 320|   USA|         3693|1970-01-01|
|        11.0|        8|       318.0|       150|            18.0|  plymouth satellite|   USA|         3436|1970-01-01|
+------------+---------+------------+----------+----------------+--------------------+------+-------------+----------+
only showing top 3 rows


In [5]:
cars.createOrReplaceTempView("cars")

In [6]:
# how many cars are there from USA

In [7]:
usCars1 = cars.where(F.col("Origin")=="USA")
usCars1

DataFrame[Acceleration: double, Cylinders: bigint, Displacement: double, Horsepower: bigint, Miles_per_Gallon: double, Name: string, Origin: string, Weight_in_lbs: bigint, Year: string]

In [8]:
usCars1.count()

254

In [9]:
usCars2 = spark.sql('SELECT * from cars WHERE Origin = "USA"')
usCars2

DataFrame[Acceleration: double, Cylinders: bigint, Displacement: double, Horsepower: bigint, Miles_per_Gallon: double, Name: string, Origin: string, Weight_in_lbs: bigint, Year: string]

In [10]:
usCars2.count()

254

In [11]:
usCars3 = spark.sql('SELECT COUNT(*) from cars WHERE Origin = "USA"')
usCars3

DataFrame[count(1): bigint]

In [12]:
usCars3.show()

+--------+
|count(1)|
+--------+
|     254|
+--------+



In [13]:
usCars3.first()[0]

254

In [14]:
# report the number of cars in each country of origin
carsPerCountry1 = cars.groupBy(F.col("Origin")).count()
carsPerCountry1.show()

+------+-----+
|Origin|count|
+------+-----+
|Europe|   73|
|   USA|  254|
| Japan|   79|
+------+-----+



In [15]:
carsPerCountry2 = spark.sql("SELECT Origin, COUNT(*) FROM cars GROUP BY Origin")
carsPerCountry2.show()

+------+--------+
|Origin|count(1)|
+------+--------+
|Europe|      73|
|   USA|     254|
| Japan|      79|
+------+--------+



In [16]:
# report the distinct name of cars in title case
def f(s:str) -> str:
    return str(s).title() if s else None
    
title_udf = F.udf(f, T.StringType()) # create a spark user define function
title_udf

/usr/local/lib/python3.12/site-packages/pyspark/sql/udf.py:134: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


<function __main__.f(s: str) -> str>

In [17]:
d1 = cars.select(title_udf(F.col("Name"))).distinct()
d1.show()

+--------------------+
|             f(Name)|
+--------------------+
|        Datsun Pl510|
|Volkswagen 1131 D...|
|         Amc Gremlin|
|  Amc Rebel Sst (Sw)|
|           Fiat X1.9|
|Chevrolet Monte C...|
|     Amc Concord D/L|
|       Honda Prelude|
|      Toyota Mark Ii|
|      Chevrolet Nova|
|       Amc Pacer D/L|
|       Datsun 280-Zx|
|Toyota Corolla 16...|
|   Volkswagen Rabbit|
|     Ford Torino 500|
| Honda Civic 1500 Gl|
|Plymouth Valiant ...|
|Citroen Ds-21 Pallas|
|Oldsmobile Cutlas...|
| Dodge Challenger Se|
+--------------------+
only showing top 20 rows


In [18]:
# register the udf for SQL
spark.udf.register("title", f, T.StringType())

<function __main__.f(s: str) -> str>

In [19]:
d2 = spark.sql("SELECT DISTINCT title(Name) FROM cars")
d2.show()

+--------------------+
|         title(Name)|
+--------------------+
|        Datsun Pl510|
|Volkswagen 1131 D...|
|         Amc Gremlin|
|  Amc Rebel Sst (Sw)|
|           Fiat X1.9|
|Chevrolet Monte C...|
|     Amc Concord D/L|
|       Honda Prelude|
|      Toyota Mark Ii|
|      Chevrolet Nova|
|       Amc Pacer D/L|
|       Datsun 280-Zx|
|Toyota Corolla 16...|
|   Volkswagen Rabbit|
|     Ford Torino 500|
| Honda Civic 1500 Gl|
|Plymouth Valiant ...|
|Citroen Ds-21 Pallas|
|Oldsmobile Cutlas...|
| Dodge Challenger Se|
+--------------------+
only showing top 20 rows


In [20]:
# note in this example Spark has a similar function initcap
d3 = spark.sql("SELECT DISTINCT initcap(Name) FROM cars")
d3.show()

+--------------------+
|       initcap(Name)|
+--------------------+
|        Datsun Pl510|
|Volkswagen 1131 D...|
|         Amc Gremlin|
|           Fiat X1.9|
|Chevrolet Monte C...|
|       Honda Prelude|
|      Toyota Mark Ii|
|      Chevrolet Nova|
|   Volkswagen Rabbit|
|     Ford Torino 500|
| Honda Civic 1500 Gl|
|Plymouth Valiant ...|
|Citroen Ds-21 Pallas|
|Oldsmobile Cutlas...|
| Dodge Challenger Se|
|           Maxda Rx3|
|Fiat 124 Sport Coupe|
|     Volvo 145e (sw)|
|Chevrolet Caprice...|
|            Audi Fox|
+--------------------+
only showing top 20 rows


In [22]:
# writing the results, write a table in the local warehouse directory
cars.write.saveAsTable("cars1")

In [23]:
spark.stop()